In [ ]:
import os
%load_ext autoreload
%autoreload 2

In [ ]:
# Cell 1: Imports (Make sure these are at the top of your notebook)
import pandas as pd
import numpy as np
import torch
import gc
import os
from preprocessing.preprocessing import preprocess_lob_data, split_data_chronological, normalize_features
from preprocessing.dataset import create_sequences, LOBSequenceDataset
from training.trainer import Trainer, Config

In [ ]:
print(os.getcwd())

In [ ]:
os.chdir('..')  # only need this temporily to get the right paths, change as needed
print(os.getcwd())

In [ ]:
import yaml
with open("configs/test.cfg", "r") as f:
    config_dict = yaml.safe_load(f)
config = Config(config_dict)

In [ ]:
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# --- 1. Preprocess Data ---
df_processed = preprocess_lob_data(config)
gc.collect()

In [ ]:
import matplotlib.pyplot as plt
class_counts = df_processed['label'].value_counts()
# Plot
class_counts.plot(kind='bar')
plt.title('Class Balance')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

In [ ]:
# --- 2. Split Data ---
train_df, val_df, test_df = split_data_chronological(
    df_processed,
    train_frac=config_dict['split_fractions']['train'],
    val_frac=config_dict['split_fractions']['val'],
    test_frac=config_dict['split_fractions']['test']
)
del df_processed
gc.collect()

In [ ]:
# --- 3. Normalize Features ---

# Select ONLY numeric feature columns for scaling
# Exclude the 'label' column as well
numeric_cols = train_df.select_dtypes(include=np.number).columns
feature_cols = [col for col in numeric_cols if col != 'label']

# Optional: Print to verify which columns are being scaled
print(f"Columns selected for scaling ({len(feature_cols)}): {feature_cols[:10]}...") # Print first 10

# Check if feature_cols is empty
if not feature_cols:
    raise ValueError("No numerical feature columns found for scaling after excluding 'label'.")

# Call normalize_features with the correctly filtered list
train_df_norm, val_df_norm, test_df_norm, scaler = normalize_features(
    train_df, val_df, test_df, feature_cols
)
train_df_norm.loc[:, "label"] = train_df.loc[:, "label"].copy()
val_df_norm.loc[:, "label"] = val_df.loc[:, "label"].copy()
test_df_norm.loc[:, "label"] = test_df.loc[:, "label"].copy()
del train_df, val_df, test_df
gc.collect()


In [ ]:
# --- 4. Create Sequences ---
# Separate features and labels before sequencing
y_train_series = train_df_norm.pop('label').values
X_train_features = train_df_norm[feature_cols].astype(np.float16).values

y_val_series = val_df_norm.pop('label').values
X_val_features = val_df_norm[feature_cols].astype(np.float16).values

y_test_series = test_df_norm.pop('label').values
X_test_features = test_df_norm[feature_cols].astype(np.float16).values

T = config.sequence_length
X_train_seq, y_train_seq = create_sequences(X_train_features, y_train_series, T, precision=np.float32)
X_val_seq, y_val_seq = create_sequences(X_val_features, y_val_series, T, precision=np.float32)
X_test_seq, y_test_seq = create_sequences(X_test_features, y_test_series, T, precision=np.float32)

print("\nSequence Creation Complete:")
print(f"X_train_seq shape: {X_train_seq.shape}, y_train_seq shape: {y_train_seq.shape}")
print(f"X_val_seq shape: {X_val_seq.shape}, y_val_seq shape: {y_val_seq.shape}")
print(f"X_test_seq shape: {X_test_seq.shape}, y_test_seq shape: {y_test_seq.shape}")

del train_df_norm, val_df_norm, test_df_norm # Free memory
del X_train_features, y_train_series, X_val_features, y_val_series, X_test_features, y_test_series
gc.collect()

In [ ]:
# --- 5. Create Datasets & DataLoaders ---
train_dataset = LOBSequenceDataset(X_train_seq, y_train_seq, device=device)
val_dataset = LOBSequenceDataset(X_val_seq, y_val_seq, device=device)
test_dataset = LOBSequenceDataset(X_test_seq, y_test_seq, device=device)

#train_loader = DataLoader(train_dataset, batch_size=config.train.batch_size, shuffle=True) # num_workers/pin_memory optional
#val_loader = DataLoader(val_dataset, batch_size=config.train.batch_size, shuffle=False)
#test_loader = DataLoader(test_dataset, batch_size=config.train.batch_size, shuffle=False)
# gc.collect()

In [ ]:
train_dataset.features.shape[2]

In [ ]:
config.model.in_features = 43

In [ ]:
output_dir = 'results/'
trainer = Trainer(config, train_dataset, val_dataset, test_dataset, device)

In [ ]:
trainer.train()

In [ ]:
pd.Series(trainer.train_loss).plot(label="Train loss")
pd.Series(trainer.val_loss).plot(label="validation loss")
plt.title("Loss Curve Vs Epoch")
plt.legend()
plt.show()

In [ ]:
trainer.val_score

In [ ]:
pd.Series(trainer.train_score).plot(label="train F1 score")
pd.Series(trainer.val_score).plot(label="val F1 score")
plt.title("F1 Score Vs Epoch")
plt.legend()
plt.show()

In [ ]:
trainer.output_dir = "results/"
trainer.save_model(name = "default_cnn_10_epochs.pt")